# Case Study 2: Credit Card Fraud Detection

**Course:** Machine Learning Essentials
**Student:** *(write your name and roll number here)*

**Objective:** Apply XGBoost on a heavily imbalanced transactions dataset to
detect fraudulent credit card transactions. Use SMOTE for oversampling the
minority (fraud) class, tune the decision threshold, and interpret results
using feature importance scores.

> Note: The real IEEE-CIS Fraud Detection dataset (Kaggle) is very large
> (500+ columns, needs a Kaggle account/API key to download) and can't be
> fetched directly inside this environment. So this notebook generates a
> smaller **synthetic** transactions dataset with the same key properties as
> real fraud data — heavy class imbalance (~1-2% fraud) and a similar set of
> features (amount, time, distance from home, card type, etc.). Every step
> below (SMOTE, XGBoost, threshold tuning, feature importance) works exactly
> the same way if you swap in the real `train_transaction.csv` from Kaggle —
> just change the data-loading cell in Section 2.


## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report,
                             roc_auc_score, roc_curve, precision_recall_curve,
                             precision_score, recall_score, f1_score)

from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

np.random.seed(42)


## 2. Load / Create the Transactions Dataset

Each row = one credit card transaction. Columns (simplified version of the
kind of features found in IEEE-CIS / Kaggle credit card fraud datasets):

| Column | Meaning |
|---|---|
| amount | transaction amount (in ₹ / $) |
| hour | hour of day the transaction happened (0-23) |
| distance_from_home | distance (km) between cardholder's home and transaction location |
| distance_from_last_txn | distance (km) from the previous transaction location |
| is_online | 1 if the transaction was made online, 0 if in-person (card present) |
| is_new_merchant | 1 if this merchant has never been used by this card before |
| card_present | 1 if the physical card was used (chip/swipe), 0 if card-not-present |
| num_txn_last_24h | number of transactions this card made in the last 24 hours |
| is_fraud | **target**: 1 if the transaction was fraudulent, else 0 |

To keep things realistic, only about **1.5% of transactions are fraud** —
this is the "heavily imbalanced" part of the problem.


In [ ]:
n_txn = 8000
fraud_rate = 0.015

amount = np.random.exponential(scale=60, size=n_txn).round(2)
hour = np.random.randint(0, 24, n_txn)
distance_from_home = np.random.exponential(scale=15, size=n_txn).round(2)
distance_from_last_txn = np.random.exponential(scale=10, size=n_txn).round(2)
is_online = np.random.binomial(1, 0.4, n_txn)
is_new_merchant = np.random.binomial(1, 0.2, n_txn)
card_present = 1 - is_online
num_txn_last_24h = np.random.poisson(2, n_txn)

df = pd.DataFrame({
    'amount': amount,
    'hour': hour,
    'distance_from_home': distance_from_home,
    'distance_from_last_txn': distance_from_last_txn,
    'is_online': is_online,
    'is_new_merchant': is_new_merchant,
    'card_present': card_present,
    'num_txn_last_24h': num_txn_last_24h,
})

# Build a TRUE fraud risk score from patterns that resemble real fraud signals:
# large amount, odd hours, far from home, new merchant, online, many txns/day.
risk_score = (
    0.015 * (df['amount'] - 60)
    + 0.06 * df['distance_from_home']
    + 0.05 * df['distance_from_last_txn']
    + 1.1  * df['is_online']
    + 1.3  * df['is_new_merchant']
    + 0.35 * df['num_txn_last_24h']
    + 0.9  * ((df['hour'] < 5) | (df['hour'] > 22)).astype(int)   # odd hours
    - 6.5
)
prob_fraud = 1 / (1 + np.exp(-risk_score))

# scale probabilities so the overall fraud rate is close to our target (1.5%)
prob_fraud = prob_fraud * (fraud_rate / prob_fraud.mean())
prob_fraud = np.clip(prob_fraud, 0, 1)

df['is_fraud'] = np.random.binomial(1, prob_fraud)

print('Fraud rate:', round(df['is_fraud'].mean() * 100, 2), '%')
df.head()


## 3. Exploratory Data Analysis (EDA)

In [ ]:
df.info()


In [ ]:
print(df['is_fraud'].value_counts())
print()
print('Class balance (%):')
print(df['is_fraud'].value_counts(normalize=True) * 100)

sns.countplot(x='is_fraud', data=df)
plt.title('Class Imbalance: Genuine (0) vs Fraud (1) Transactions')
plt.show()


In [ ]:
plt.figure(figsize=(8,6))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()


## 4. Train/Test Split + Feature Scaling

**Important:** we split into train/test **before** applying SMOTE, and we
only oversample the **training** set. If we oversampled before splitting,
copies of the same synthetic fraud sample could leak into both train and
test sets, giving a falsely inflated (and wrong) performance score.


In [ ]:
X = df.drop(columns=['is_fraud'])
y = df['is_fraud']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('Train shape:', X_train.shape, ' Test shape:', X_test.shape)
print('Fraud cases in train:', y_train.sum(), '/', len(y_train))
print('Fraud cases in test :', y_test.sum(), '/', len(y_test))


## 5. Handling Imbalance with SMOTE

**SMOTE (Synthetic Minority Over-sampling Technique)** creates new,
*synthetic* fraud examples by interpolating between existing fraud
transactions and their nearest fraud neighbors in feature space — it does
not just copy-paste existing rows. This gives the model many more fraud
examples to learn from, instead of the ~1.5% it would otherwise see.

We apply SMOTE **only on the training set**, never on the test set (the
test set must reflect the real, imbalanced world so our evaluation is
honest).


In [ ]:
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)

print('Before SMOTE:', y_train.value_counts().to_dict())
print('After SMOTE :', pd.Series(y_train_res).value_counts().to_dict())


## 6. Train an XGBoost Classifier

XGBoost (Extreme Gradient Boosting) builds an ensemble of decision trees,
where each new tree tries to correct the errors of the previous ones. It
handles non-linear feature interactions well (e.g. "online + new merchant +
odd hour" together) and is a standard choice for fraud detection.


In [ ]:
model = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.1,
    eval_metric='logloss',
    random_state=42
)
model.fit(X_train_res, y_train_res)


## 7. Evaluation

Because fraud is rare, **accuracy is a misleading metric** here (a model
that predicts "not fraud" for everything would still be ~98.5% accurate!).
So we focus on:
- **ROC-AUC** — how well the model ranks transactions by fraud probability.
- **Precision / Recall / F1** for the fraud class specifically.
- **Precision-Recall curve** — often more informative than ROC when the
  positive class (fraud) is rare.


In [ ]:
y_pred_default = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:, 1]

print('Accuracy:', round(accuracy_score(y_test, y_pred_default), 4), '  <-- misleading, see note above')
print()
print('Confusion Matrix (threshold = 0.5):')
print(confusion_matrix(y_test, y_pred_default))
print()
print('Classification Report (threshold = 0.5):')
print(classification_report(y_test, y_pred_default, digits=3))

auc = roc_auc_score(y_test, y_prob)
print('ROC-AUC Score:', round(auc, 3))


In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_prob)

plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, label=f'XGBoost (AUC = {auc:.3f})')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Random guess')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate (Recall)')
plt.title('ROC Curve - Credit Card Fraud Detection')
plt.legend()
plt.show()


### 7.1 Precision-Recall Curve

For rare-event problems like fraud, the Precision-Recall curve is usually
more informative than the ROC curve, because it focuses only on how well we
handle the minority (fraud) class, ignoring the huge number of easy true
negatives.


In [ ]:
precision, recall, pr_thresholds = precision_recall_curve(y_test, y_prob)

plt.figure(figsize=(6,5))
plt.plot(recall, precision)
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve - Fraud Detection')
plt.show()


## 8. Tuning the Decision Threshold

By default, `model.predict()` uses a threshold of 0.5 to convert a
probability into a 0/1 label. In fraud detection, missing a fraud case
(false negative) is usually far more costly than investigating a genuine
transaction that got flagged (false positive) — so banks typically **lower**
the threshold to catch more fraud, accepting some extra false alarms.

Below we scan multiple thresholds and see how precision, recall, and F1
change.


In [ ]:
results = []
for t in [0.5, 0.4, 0.3, 0.2, 0.1, 0.05]:
    y_pred_t = (y_prob >= t).astype(int)
    results.append({
        'threshold': t,
        'precision': round(precision_score(y_test, y_pred_t, zero_division=0), 3),
        'recall': round(recall_score(y_test, y_pred_t, zero_division=0), 3),
        'f1': round(f1_score(y_test, y_pred_t, zero_division=0), 3),
        'flagged_transactions': int(y_pred_t.sum())
    })

threshold_df = pd.DataFrame(results)
threshold_df


**How to read this table:** as the threshold goes down, recall goes up
(we catch more fraud) but precision goes down (more false alarms, i.e. more
genuine transactions get flagged and blocked/reviewed unnecessarily). The
"right" threshold is a business decision: it depends on the cost of
investigating a false alarm vs. the cost of a missed fraud (chargebacks,
customer trust, regulatory fines). A common practical choice is the
threshold that maximizes F1-score, or a fixed recall target (e.g. "catch at
least 90% of fraud") chosen by the risk team.


## 9. Feature Importance — Interpreting the Model

XGBoost gives us a feature importance score for each input feature, showing
how much each feature contributed to the model's decisions (based on how
often/effectively it was used to split the trees).


In [ ]:
importances = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values(by='importance', ascending=False)

importances


In [ ]:
plt.figure(figsize=(8,5))
sns.barplot(x='importance', y='feature', data=importances, palette='viridis')
plt.title('XGBoost Feature Importance - Fraud Detection')
plt.xlabel('Importance Score')
plt.show()


**Interpretation (based on how we generated the data):** features like
`is_new_merchant`, `is_online`, `num_txn_last_24h`, and `distance_from_home`
tend to matter most — this matches real-world fraud intuition: fraud is
more common on new/unfamiliar merchants, online (card-not-present)
transactions, unusually frequent transactions in a short time, and
transactions far from the cardholder's usual location.


## 10. Conclusion

- Real fraud datasets are heavily imbalanced (often <1% fraud), so accuracy
  alone is not a useful metric — we used ROC-AUC, precision, recall, and the
  precision-recall curve instead.
- **SMOTE** was applied only on the training data to give XGBoost more
  fraud examples to learn from, without leaking synthetic data into the
  test set.
- **XGBoost** was chosen because it handles non-linear feature interactions
  well and gives interpretable feature importance scores.
- **Threshold tuning** showed the trade-off between catching more fraud
  (higher recall) and generating more false alarms (lower precision) — in
  practice this threshold is chosen based on business/financial cost, not
  just left at the default 0.5.
- Feature importance showed which transaction characteristics drive fraud
  risk, which can help a bank's risk team decide what checks to add
  (e.g. extra verification for new merchants or unusual locations).

**Possible improvements (future work):**
- Use the real IEEE-CIS dataset from Kaggle (requires a Kaggle API key).
- Try SHAP values for a more detailed, per-transaction explanation instead
  of just global feature importance.
- Compare against other imbalance-handling techniques (class weights,
  undersampling, ADASYN).
